##### Table reading with SQL from catalog

In [0]:
%sql
USE CATALOG test;
USE SCHEMA sql;

In [0]:
%sql
select * from population;

country,continent,total_population
India,Asia,1428627663
China,Asia,1425671352
United States,Americas,339996564
Indonesia,Asia,277534123
Pakistan,Asia,240485658
Nigeria,Africa,223804632
Brazil,Americas,216422446
Bangladesh,Asia,172954319
Russia,Europe,144444359
Mexico,Americas,128455567


##### Table reading with PYSPARK from catalog

In [0]:
population_df = spark.read.table("test.pyspark.population")
population_df.display()

country,continent,total_population
India,Asia,1428627663
China,Asia,1425671352
United States,Americas,339996564
Indonesia,Asia,277534123
Pakistan,Asia,240485658
Nigeria,Africa,223804632
Brazil,Americas,216422446
Bangladesh,Asia,172954319
Russia,Europe,144444359
Mexico,Americas,128455567


##### Find the highest & lowest population of each continent

###### Using CTE SQL & CTE with Windows Function

In [0]:
%sql
with cte_population as (
    select continent, sum(total_population) as cont_population from population group by continent
),
cte_world as (
    select sum(total_population) as world_population from population
)
select 
    a.country, 
    a.continent, 
    a.total_population,
    b.cont_population,
    c.world_population
from population as a
join cte_population as b
join cte_world as c
on a.continent = b.continent;

country,continent,total_population,cont_population,world_population
India,Asia,1428627663,3545273115,4598396683
China,Asia,1425671352,3545273115,4598396683
United States,Americas,339996564,684874577,4598396683
Indonesia,Asia,277534123,3545273115,4598396683
Pakistan,Asia,240485658,3545273115,4598396683
Nigeria,Africa,223804632,223804632,4598396683
Brazil,Americas,216422446,684874577,4598396683
Bangladesh,Asia,172954319,3545273115,4598396683
Russia,Europe,144444359,144444359,4598396683
Mexico,Americas,128455567,684874577,4598396683


In [0]:
%sql
----------  Using legacy CTE with SQL  ------------
-- select country, continent, total_population, cont_population from 
-- (
--     select 
--         *,
--         sum(total_population) over(
--             partition by continent
--         ) as cont_population,
--         dense_rank() over(
--             partition by continent
--             order by total_population desc
--         ) as highest_population_per_continent,
--         dense_rank() over(
--             partition by continent
--             order by total_population asc
--         ) as lowest_population_per_continent
--     from population
-- )
-- where highest_population_per_continent = 1 or lowest_population_per_continent = 1;


----------- Using CTE with windows Function ----------------
with cte as (
        select 
        *,
        sum(total_population) over(
            partition by continent
        ) as cont_population,
        dense_rank() over(
            partition by continent
            order by total_population desc
        ) as highest_population_per_continent,
        dense_rank() over(
            partition by continent
            order by total_population asc
        ) as lowest_population_per_continent
    from population
)
select 
    continent,
    country,
    total_population,
    cont_population
from cte
WHERE highest_population_per_continent = 1
   OR lowest_population_per_continent = 1;

continent,country,total_population,cont_population
Africa,Nigeria,223804632,223804632
Americas,Mexico,128455567,684874577
Americas,United States,339996564,684874577
Asia,Bangladesh,172954319,3545273115
Asia,India,1428627663,3545273115
Europe,Russia,144444359,144444359


###### Using Pyspark with windows function

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, dense_rank, col, desc, asc

population_df.withColumn(
    "cont_population",
    sum(col("total_population")).over(
        Window.partitionBy(col("continent"))
    )
).withColumn(
    "highest_population_per_continent",
    dense_rank().over(
        Window.partitionBy(col("continent"))
                .orderBy(col("total_population").desc())
    )
).withColumn(
    "lowest_population_per_continent",
    dense_rank().over(
        Window.partitionBy(col("continent"))
                .orderBy(col("total_population").asc())
    )
).filter(
    (col("highest_population_per_continent") == 1) |
    (col("lowest_population_per_continent") == 1)
).select(
    col("country"),
    col("continent"),
    col("total_population"),
    col("cont_population")
).display()

country,continent,total_population,cont_population
Nigeria,Africa,223804632,223804632
Mexico,Americas,128455567,684874577
United States,Americas,339996564,684874577
Bangladesh,Asia,172954319,3545273115
India,Asia,1428627663,3545273115
Russia,Europe,144444359,144444359


##### Please write a SQL statement using CTE to query the table population with below columns:
###### - country
###### - continent
###### - cnty_population - Population of the country
###### - highest_cont_population - Population of the highest populated country in the continent
###### - highest_world_population - Population of the highest populated country in the entire table

In [0]:
%sql
------------------- Using CTE for simple subquery -----------------------
-- with cte_cont as (
--     select
--         continent,
--         max(total_population) as highest_cont_population
--     from population 
--     group by continent
-- ),
-- cte_world as (
--     select 
--         max(total_population) as highest_world_population 
--     from population
-- )
-- select
--     a.country,
--     a.continent,
--     a.total_population as cnty_population,
--     cte_cont.highest_cont_population,
--     w.highest_world_population
-- from population a 
-- join cte_cont on a.continent = cte_cont.continent
-- join cte_world as w


------------------- Using Windows function -----------------------
select 
    country,
    continent,
    total_population as cnty_population,
    max(total_population) over (
        partition by continent
    ) as highest_cont_population,
    max(total_population) over () as highest_world_population
from population
order by country;

country,continent,cnty_population,highest_cont_population,highest_world_population
Bangladesh,Asia,172954319,1428627663,1428627663
Brazil,Americas,216422446,339996564,1428627663
China,Asia,1425671352,1428627663,1428627663
India,Asia,1428627663,1428627663,1428627663
Indonesia,Asia,277534123,1428627663,1428627663
Mexico,Americas,128455567,339996564,1428627663
Nigeria,Africa,223804632,223804632,1428627663
Pakistan,Asia,240485658,1428627663,1428627663
Russia,Europe,144444359,144444359,1428627663
United States,Americas,339996564,339996564,1428627663


######  Solving using PYSPARK - Windows function

In [0]:
from pyspark.sql.functions import max

population_df.withColumn(
    "highest_cont_population",
    max(col("total_population")).over(
        Window.partitionBy(col("continent"))
    )
).withColumn(
    "highest_world_population",
    max(col("total_population")).over(
        Window.partitionBy()
    ).alias("highest_world_population")
).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


country,continent,total_population,highest_cont_population,highest_world_population
India,Asia,1428627663,1428627663,1428627663
China,Asia,1425671352,1428627663,1428627663
United States,Americas,339996564,339996564,1428627663
Indonesia,Asia,277534123,1428627663,1428627663
Pakistan,Asia,240485658,1428627663,1428627663
Nigeria,Africa,223804632,223804632,1428627663
Brazil,Americas,216422446,339996564,1428627663
Bangladesh,Asia,172954319,1428627663,1428627663
Russia,Europe,144444359,144444359,1428627663
Mexico,Americas,128455567,339996564,1428627663


##### Using CTE to query the table population, and produce the result set as shown below:
###### - country
###### - continent
###### - cnty_population - Population of the country
###### - cont_population - Total population of the continent to which the country belongs to
###### - highest_cont_population - Population of the continent with the highest population in the entire table

In [0]:
%sql
-- with cte as (
--     select
--         continent,
--         sum(total_population) as cont_population
--     from population
--     group by continent
-- ),
-- cte_max as (
--     select 
--         max(cont_population) as highest_cont_population
--     from cte
-- )
-- select 
--     a.country,
--     a.continent,
--     a.total_population as cnty_population,
--     b.cont_population,
--     c.highest_cont_population
-- from population a
-- join cte b on a.continent = b.continent
-- join cte_max c;


select 
    country,
    continent,
    total_population as cnty_population,
    sum(total_population) over (
        partition by continent
    ) as cont_population,
    max(
        sum(total_population) over (
            partition by continent
        )
    ) over () as highest_cont_population
from population;



country,continent,cnty_population,cont_population,highest_cont_population
India,Asia,1428627663,3545273115,3545273115
China,Asia,1425671352,3545273115,3545273115
United States,Americas,339996564,684874577,3545273115
Indonesia,Asia,277534123,3545273115,3545273115
Pakistan,Asia,240485658,3545273115,3545273115
Nigeria,Africa,223804632,223804632,3545273115
Brazil,Americas,216422446,684874577,3545273115
Bangladesh,Asia,172954319,3545273115,3545273115
Russia,Europe,144444359,144444359,3545273115
Mexico,Americas,128455567,684874577,3545273115


######  Solving using PYSPARK - Windows function

In [0]:
population_df.withColumn(
    "cont_population",
    sum(col("total_population")).over(
        Window.partitionBy(col("continent"))
    )
).withColumn(
    "highest_cont_population",
    max(col("cont_population")).over(
        Window.partitionBy()
    )
).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


country,continent,total_population,cont_population,highest_cont_population
India,Asia,1428627663,3545273115,3545273115
China,Asia,1425671352,3545273115,3545273115
United States,Americas,339996564,684874577,3545273115
Indonesia,Asia,277534123,3545273115,3545273115
Pakistan,Asia,240485658,3545273115,3545273115
Nigeria,Africa,223804632,223804632,3545273115
Brazil,Americas,216422446,684874577,3545273115
Bangladesh,Asia,172954319,3545273115,3545273115
Russia,Europe,144444359,144444359,3545273115
Mexico,Americas,128455567,684874577,3545273115


In [0]:
%sql
create view vw_population
as
select * from population;

In [0]:
%sql
select * from vw_population;

country,continent,total_population
India,Asia,1428627663
China,Asia,1425671352
United States,Americas,339996564
Indonesia,Asia,277534123
Pakistan,Asia,240485658
Nigeria,Africa,223804632
Brazil,Americas,216422446
Bangladesh,Asia,172954319
Russia,Europe,144444359
Mexico,Americas,128455567


In [0]:
%sql
drop view vw_population;